# 30_stable_diffusion.ipynb

**14주차 · 2교시** 실습 노트북

- 이론 설명과 관찰 포인트는 배포 자료(`14week/student/`)를 함께 보세요.
- 실행 환경: `%DL2026_HOME%\venv` 활성화 후 `Python (dl2026)` 커널.
- 전체 11셀. 위에서부터 순서대로 실행합니다.

## 1. 실습 3 — 첫 이미지 생성 ★

**셀 1** — 환경·캐시 확인부터 ★

In [ ]:
import os, time, torch
print("HF_HOME :", os.environ.get("HF_HOME"))
print("GPU     :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
if torch.cuda.is_available():
    print("VRAM    :", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), "GB")
# os.environ["HF_HUB_OFFLINE"] = "1"        # ★ 캐시만 쓰게 강제 (다운로드 방지)

**셀 2** — 파이프라인 로딩 (8GB 필수 옵션 ★★)

In [ ]:
from diffusers import AutoPipelineForText2Image

MODEL = "stabilityai/sd-turbo"                  # ★ 1순위. 캐시에 있는 것으로
pipe = AutoPipelineForText2Image.from_pretrained(
    MODEL,
    torch_dtype=torch.float16,                  # ★ fp16 — VRAM 절반 (7주차 AMP)
    variant="fp16",
).to("cuda")
pipe.enable_attention_slicing()                 # ★ 어텐션을 조각내 계산 → 피크 VRAM ↓
pipe.set_progress_bar_config(disable=False)

print("구성 요소 :", [k for k in pipe.components.keys()])   # ★ §0 의 세 부품이 보인다

> **관찰 포인트 ★**: `pipe.components` 에 `text_encoder`, `unet`, `vae` 가 **그대로 보입니다.** §0 에서 그린 그림과 대조해 보세요. **추상적인 설명이 아니라 실제 객체**입니다.

**셀 3** — 첫 생성 ★

In [ ]:
prompt = "a cozy wooden cabin in a snowy forest, warm light, digital art"

g = torch.Generator("cuda").manual_seed(42)     # ★ seed 고정 (6주차 재현성)
t0 = time.time()
img = pipe(prompt, num_inference_steps=4, guidance_scale=0.0,   # SD-Turbo 권장값
           generator=g).images[0]
print(f"생성 시간 : {time.time()-t0:.1f}초")
print(f"피크 VRAM : {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

os.makedirs("outputs/generated", exist_ok=True)
img.save("outputs/generated/first.png")
img

**셀 4** — OOM 이 났다면 ★ 대응 순서

In [ ]:
"""
① pipe.enable_attention_slicing()          ← 이미 적용
② pipe.enable_vae_slicing()                ← VAE 디코딩을 조각내기
③ height=384, width=384                    ← 크기 축소 (512 미만은 품질 저하)
④ pipe.enable_model_cpu_offload()          ← 안 쓰는 부품을 CPU 로 (느려짐)
⑤ 그래도 안 되면 → Colab(T4) 로 전환       ★ 즉시 판단
"""
torch.cuda.empty_cache()                    # 커널 재시작 없이 정리
print("VRAM 정리 후 :", round(torch.cuda.memory_allocated()/1e9, 2), "GB")

## 2. 실습 4 — 8GB 대응 옵션 비교

**셀 5** — 측정 함수

In [ ]:
def bench(pipe, prompt, steps, seed=42, tag=""):
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    g = torch.Generator("cuda").manual_seed(seed)
    t0 = time.time()
    im = pipe(prompt, num_inference_steps=steps, guidance_scale=0.0, generator=g).images[0]
    dt = time.time() - t0
    vram = torch.cuda.max_memory_allocated() / 1e9
    print(f"{tag:28s} | step {steps:3d} | {dt:5.1f}초 | 피크 VRAM {vram:.2f} GB")
    return im, dt, vram

**셀 6** — ① slicing 유무 비교

In [ ]:
pipe.disable_attention_slicing()
bench(pipe, prompt, 4, tag="fp16, slicing 없음")

pipe.enable_attention_slicing()
bench(pipe, prompt, 4, tag="fp16, slicing 있음")

**셀 7** — ② step 수별 비교 ★

In [ ]:
for s in [1, 2, 4, 8]:
    bench(pipe, prompt, s, tag=f"step {s}")

## 3-1. step 수와 품질

**셀 8** — 같은 seed, 다른 step ★

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "Malgun Gothic"; plt.rcParams["axes.unicode_minus"] = False

steps = [1, 2, 4, 8]
imgs = [pipe(prompt, num_inference_steps=s, guidance_scale=0.0,
             generator=torch.Generator("cuda").manual_seed(42)).images[0] for s in steps]

fig, axes = plt.subplots(1, len(steps), figsize=(4 * len(steps), 4.4))
for a, im, s in zip(axes, imgs, steps):
    a.imshow(im); a.axis("off"); a.set_title(f"step={s}")
plt.suptitle("같은 프롬프트 · 같은 seed — step 수만 다르다")
plt.tight_layout(); plt.show()

> **관찰 포인트 ★**: step 이 늘수록 **세부가 정리됩니다.** 다만 SD-Turbo 는 **4 스텝 이후 거의 좋아지지 않습니다** — 적은 스텝에 맞춰 증류된 모델이기 때문입니다. *"무조건 많이 하면 좋다"* 가 아니라 **모델마다 적정 구간이 있다**는 감각이 목표입니다.

## 3-2. seed 와 재현성 ★

**셀 9** — 같은 seed = 같은 이미지 ★★

In [ ]:
import numpy as np
a = pipe(prompt, num_inference_steps=4, guidance_scale=0.0,
         generator=torch.Generator("cuda").manual_seed(7)).images[0]
b = pipe(prompt, num_inference_steps=4, guidance_scale=0.0,
         generator=torch.Generator("cuda").manual_seed(7)).images[0]
c = pipe(prompt, num_inference_steps=4, guidance_scale=0.0,
         generator=torch.Generator("cuda").manual_seed(8)).images[0]

print("seed 7 두 번이 같은가 :", np.array_equal(np.array(a), np.array(b)), " ← True ★")
print("seed 7 과 8 이 같은가 :", np.array_equal(np.array(a), np.array(c)), " ← False")

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for ax, im, t in zip(axes, [a, b, c], ["seed 7", "seed 7 (재실행)", "seed 8"]):
    ax.imshow(im); ax.axis("off"); ax.set_title(t)
plt.tight_layout(); plt.show()

> **핵심 ★★ (기말 출제 지점)**: **같은 seed 가 같은 이미지를 만드는 이유**는 **시작 잡음이 같기 때문**입니다(1교시 §3-2). 확산은 그 잡음을 결정론적으로 지워 갑니다. **6주차 재현성**과 **완전히 같은 원리**이고, 13주차 GAN 의 `z` 고정과도 같습니다.

## 3-3. 프롬프트 실험

**셀 10** — 프롬프트를 바꿔 가며 (자유 실험)

In [ ]:
prompts = [
    "a cabin in a forest",                                          # 짧게
    "a cozy wooden cabin in a snowy forest, warm window light",      # 구체적으로
    "a cozy wooden cabin in a snowy forest, warm window light, "
    "watercolor painting, soft pastel colors",                       # 스타일 추가
]
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
records = []
for ax, p in zip(axes, prompts):
    im = pipe(p, num_inference_steps=4, guidance_scale=0.0,
              generator=torch.Generator("cuda").manual_seed(42)).images[0]
    ax.imshow(im); ax.axis("off"); ax.set_title(p[:28] + "…", fontsize=9)
    records.append({"prompt": p, "seed": 42, "steps": 4, "model": MODEL})
plt.tight_layout(); plt.show()

**셀 11** — 생성 기록을 남긴다 (과제 제출물 ★)

In [ ]:
import json
for i, r in enumerate(records):
    with open(f"outputs/generated/meta_{i}.json", "w", encoding="utf-8") as f:
        json.dump(r, f, ensure_ascii=False, indent=2)
print("프롬프트·seed·step·모델명을 함께 저장했다 ★")

> **관찰 포인트**: 프롬프트가 **구체적일수록** 결과가 의도에 가까워집니다. 다만 **너무 길면 뒤쪽이 무시**됩니다 — 텍스트 인코더의 입력이 **77 토큰**으로 제한되기 때문입니다. *"12주차의 `max_length` 이야기가 여기서도 나옵니다."*